# C2-linear-models — Session 1: Linear Regression and the MSE Loss

*One class session, roughly 85 minutes. Prerequisites: F3-matrices (matrices
as maps, matrix–vector action, matrix multiplication) and, through it,
F2-vectors (dot products, norms, orthogonality, projection) and
F1-scientific-python (broadcasting, axis sums, seeded randomness,
matplotlib); F4-multivar-calculus (partials, gradients, chain rule,
sum-of-squares gradients); C1-ml-fundamentals (train-test-split,
overfitting).*

**This session:** the linear prediction model in component form,
$\hat y_i = \sum_k X_{ik} w_k + b$, and the same model seen through F3's
matrix-map lens; the residual vector and the **mean squared error** (MSE) —
F4's sum-of-squares score with a name and a $1/n$; how to *evaluate* a
candidate rule at given weights and how to *differentiate* the loss at
given weights (this unit never fits anything — `C3-gradient-descent` owns
the machinery that moves weights); the residual geometry that F2's
projection picture explains; and a fully worked exam-style normal-form
multiple-choice example.

Try every checkpoint by hand first, then verify with NumPy.
Answers are collected at the end of this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. From Data to a Prediction Rule

**Motivation.**
C1 sorted learning tasks by what they predict: classification predicts a
*label*, **regression** predicts a *number* — tomorrow's temperature, a
house's sale price, a battery's remaining minutes.
This unit builds the simplest serious regression model there is, the one
every later model in this course elaborates: a **linear model**.

**The cast, in this course's fixed notation.**

- `X` — the data grid, shape $(n, d)$: $n$ rows (one per example), $d$
  columns (one per feature). $X_{ik}$ is feature $k$ of example $i$.
- `y` — the targets, shape $(n,)$: the true numbers we want to predict.
- `w` — the **weights**, shape $(d,)$: one adjustable number per feature.
- `b` — the **bias**, a single scalar: the model's baseline output.

**Definition (the linear model, component form).**
The prediction for example $i$ is
$$\hat y_i \;=\; \sum_{k=0}^{d-1} X_{ik}\, w_k \;+\; b .$$
Read it aloud: *multiply each feature by its weight, add them up, shift
by the bias.*
Weight $w_k$ says how much one unit of feature $k$ moves the prediction;
$b$ is what the model outputs when every feature is $0$.

**Worked example (by hand).**
Three examples, two features:
$$X = \begin{pmatrix} 2 & 1 \\ 1 & 3 \\ 0 & 2 \end{pmatrix},
\qquad w = (3, -1), \qquad b = 2 .$$

- $\hat y_0 = 2 \cdot 3 + 1 \cdot (-1) + 2 = 7$
- $\hat y_1 = 1 \cdot 3 + 3 \cdot (-1) + 2 = 2$
- $\hat y_2 = 0 \cdot 3 + 2 \cdot (-1) + 2 = 0$

So $\hat y = (7, 2, 0)$.

**In code, the exam-safe way.**
The exam's implementation tasks routinely ban `@`, `np.matmul`, `np.dot`,
`.T`, and loops (F3/F4's register), so we build the habit now:
`X * w` broadcasts $(n, d) \times (d,) \to (n, d)$ — each row multiplied
elementwise by the weights — and `.sum(axis=1)` collapses each row's
$k$-sum.
Adding the scalar `b` broadcasts over all $n$ entries.

In [ ]:
X_tiny = np.array([[2.0, 1.0],
                   [1.0, 3.0],
                   [0.0, 2.0]])
w_tiny = np.array([3.0, -1.0])
b_tiny = 2.0


def predict(X, w, b):
    """Linear-model predictions, shape (n,). No @, no np.dot, no loops."""
    return (X * w).sum(axis=1) + b


print("predictions:", predict(X_tiny, w_tiny, b_tiny))   # (7, 2, 0)

### Checkpoint 1

1. By hand, same $w = (3, -1)$ and $b = 2$: what does the model predict
   for a new example with features $(1, 4)$?
2. `X` has shape `(200, 6)`.
   What are the shapes of `w`, of `b`, and of the prediction array
   $\hat y$?
3. In one sentence: what does the model do with feature $k$ when
   $w_k = 0$, and why will that innocent-looking fact matter a great deal
   in Session 2?

## 2. The F3 View: One Model, Three Pictures

**Motivation.**
The component form is what you compute with.
But the same model, seen through F3's matrix-map lens, explains
*geometrically* everything this unit does later — especially why the best
rule's misses point the way they do (Section 5).

**Picture 1 — the row picture (F2).**
Row $i$ of $X$ is a vector $x_i$ with $d$ entries, and
$$\hat y_i = x_i \cdot w + b :$$
each prediction is a **dot product** of that example's features with the
weight vector, shifted by $b$.
One dot product per example — this is exactly how `(X * w).sum(axis=1)`
computes it.

**Picture 2 — the map picture (F3).**
Freeze the data and look at all $n$ predictions at once:
$$\hat y = X w + b\,\mathbf{1},$$
where $\mathbf 1$ is the all-ones vector with $n$ entries.
The matrix $X$ acts as a machine that eats the weight vector
$w \in \mathbb{R}^d$ and emits a vector of $n$ row-dot-products — F3's
matrix–vector action, nothing more.
Turning the weight knobs $w$ steers the output of a fixed linear map.

**Picture 3 — the column picture (F3's column secret).**
The same product, read column-wise: with $c_k$ = column $k$ of $X$,
$$\hat y = w_0\, c_0 + w_1\, c_1 + \dots + w_{d-1}\, c_{d-1} + b\,\mathbf 1 .$$
The prediction vector is a **blend of the feature columns plus the
all-ones column**, with the weights and bias as blend coefficients.
Whatever knobs you turn, $\hat y$ can only ever be such a blend: the
**reachable set** of prediction vectors is spanned by $d + 1$ vectors
inside $\mathbb{R}^n$.
With $n$ much larger than $d$, almost every target vector $y$ is *not*
reachable — a perfect fit is generally impossible, and the game becomes
*getting close*.
Hold that thought for Section 5.

**Geometry when $d = 1$.**
One feature: $\hat y = w_0 x + b$ is a line — $w_0$ the slope, $b$ the
intercept.
Two features: a plane over the feature plane.
The code draws one feature's worth of data with two candidate lines —
same model family, different knob settings.

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
x1 = rng.uniform(0, 10, 40)
y1 = 1.8 * x1 + 4.0 + rng.normal(0, 2.0, 40)

xs = np.linspace(0, 10, 100)
plt.figure(figsize=(6.5, 3.8))
plt.scatter(x1, y1, s=18, label="data")
plt.plot(xs, 1.8 * xs + 4.0, label=r"$\hat y = 1.8x + 4$ (good knobs)")
plt.plot(xs, 0.8 * xs + 9.0, "--", label=r"$\hat y = 0.8x + 9$ (bad knobs)")
plt.xlabel("feature x")
plt.ylabel("target y")
plt.title("One model family, two knob settings")
plt.legend()
plt.show()

### Checkpoint 2

1. For $d = 1$, write the model in slope–intercept form and say which
   symbol plays the slope and which the intercept.
2. Column picture, by hand: $X$ has columns $c_0 = (1, 0)$ and
   $c_1 = (0, 2)$, with $w = (2, 3)$ and $b = 1$.
   Write $\hat y$ as a blend of $c_0$, $c_1$, and $\mathbf 1 = (1, 1)$,
   and evaluate it.
3. One sentence: with $n = 100$ examples and $d = 2$ features, why can't
   the model hit an arbitrary target vector $y$ exactly, in the column
   picture's language?

## 3. Misses, Residuals, and the MSE

**Motivation.**
A candidate rule is only as good as its misses.
We need one number that scores how far the predictions sit from the
targets — a **loss**.
This unit's loss is the mean squared error, and it is exactly F4's
sum-of-squares score wearing its machine-learning name.

**Definition (residual).**
The **residual** of example $i$ is the signed miss
$$r_i \;=\; \hat y_i - y_i$$
— positive when the model predicts *high*, negative when it predicts
*low*.
(F4 wrote its score with the opposite convention, $y_i - \hat y_i$;
the squares below can't tell the difference, but derivatives can — we
fix $r_i = \hat y_i - y_i$ for this unit and the next, and Section 5
shows exactly where the choice shows up.)

**Definition (mean squared error).**
$$\mathrm{MSE} \;=\; \frac{1}{n} \sum_{i=0}^{n-1} \big(\hat y_i - y_i\big)^2
\;=\; \frac{1}{n} \sum_i r_i^2 .$$
Square every miss, average.
Squaring does two jobs: misses in opposite directions can't cancel, and
big misses are punished far more than proportionally (one miss of $4$
costs as much as sixteen misses of $1$).
The $1/n$ makes scores comparable across datasets of different sizes.

**Worked example.**
Section 1's predictions $\hat y = (7, 2, 0)$ against targets
$y = (6, 3, 1)$:
$$r = (1, -1, -1), \qquad
\mathrm{MSE} = \tfrac{1}{3}\big(1 + 1 + 1\big) = 1 .$$

**The F2 view.**
Stack the misses into the residual *vector* $r = \hat y - y$ in
$\mathbb{R}^n$.
Then
$$\mathrm{MSE} = \frac{\lVert r \rVert^2}{n} :$$
up to the $1/n$, the loss is the **squared distance** between the
prediction vector and the target vector.
"Make the loss small" literally means "move the reachable blend
$\hat y$ as close to $y$ as the knobs allow."
(If you want the miss in the target's own units, $\sqrt{\mathrm{MSE}}$ —
the *root* mean squared error — is the conventional report.)

In [ ]:
def mse(X, y, w, b):
    """Mean squared error of the linear model at the GIVEN weights."""
    r = predict(X, w, b) - y          # residuals, shape (n,)
    return np.mean(r**2)


y_tiny = np.array([6.0, 3.0, 1.0])
r_tiny = predict(X_tiny, w_tiny, b_tiny) - y_tiny
print("residuals:", r_tiny)                       # (1, -1, -1)
print("MSE      :", mse(X_tiny, y_tiny, w_tiny, b_tiny))   # 1.0
print("||r||^2/n:", np.sum(r_tiny**2) / r_tiny.shape[0])   # same number

### Checkpoint 3

1. By hand: $\hat y = (4, 1)$, $y = (2, 2)$. Compute the residual vector
   and the MSE.
2. Two sentences: the two reasons the definition squares the residuals
   instead of just averaging them.
3. Express the MSE in F2 vocabulary — one formula involving a norm.

## 4. Candidate Rules on a Scoreboard

**Motivation.**
This unit's central discipline — and the exam's — is **evaluating at
given weights**.
Nobody hands you a fitting routine; you are handed *candidate* weight
settings (from a teammate, from a printed table, from a problem
statement) and asked which is better and by how much.
The MSE is the scoreboard.

**The C1 rule still applies.**
Judge candidates on data they were not tuned to: split off held-out rows
(C1's train-test-split) and read *both* columns of the scoreboard.
Below: seeded data whose true recipe is
$y = 2 x_0 - x_1 + 3 + \text{noise}$, split 35 train / 15 test, and
three supplied candidates:

| candidate | $w$ | $b$ |
|---|---|---|
| A | $(2, -1)$ | $3$ |
| B | $(2.5, 0)$ | $3$ |
| C | $(1, -2)$ | $2$ |

In [ ]:
rng = np.random.default_rng(SEED)
X4 = rng.normal(0, 1, (50, 2))
y4 = 2.0 * X4[:, 0] - 1.0 * X4[:, 1] + 3.0 + rng.normal(0, 0.5, 50)

X_train, y_train = X4[:35], y4[:35]     # C1: train-test-split
X_test,  y_test  = X4[35:], y4[35:]

candidates = {
    "A": (np.array([2.0, -1.0]), 3.0),
    "B": (np.array([2.5,  0.0]), 3.0),
    "C": (np.array([1.0, -2.0]), 2.0),
}

print(f"{'cand':<6}{'train MSE':>12}{'test MSE':>12}")
for name, (w_c, b_c) in candidates.items():
    print(f"{name:<6}{mse(X_train, y_train, w_c, b_c):>12.4f}"
          f"{mse(X_test, y_test, w_c, b_c):>12.4f}")

Candidate A — the one matching the true recipe — wins both columns, and
its MSE ($\approx 0.35$–$0.40$) sits near the noise level
($0.5^2 = 0.25$): even the *true* recipe cannot score below the noise it
cannot predict.

Here the two columns agree on the ranking.
They do not always: a candidate tuned hard to the training rows can top
the train column and sink in the test column — C1's **overfitting**, on a
number scoreboard instead of an accuracy one.
Session 2 opens with exactly such a case and builds this unit's remedy.

### Checkpoint 4

1. Candidate D is the laziest possible rule: $w = (0, 0)$,
   $b = \bar y_{\text{train}}$ (the mean of the training targets).
   What does it predict for every example?
   Add it to the scoreboard in code and report both its MSEs.
2. One sentence (C1 recap): why must the scoreboard include held-out
   rows at all?
3. True or false, with one sentence: the candidate with the lowest
   *train* MSE always has the lowest *test* MSE.

## 5. The Gradient of the MSE at Given Weights

**Motivation.**
The scoreboard says *how good* a candidate is.
The **gradient** says how *sensitive* the score is to each knob — which
weight, nudged, would change the loss fastest, and in which direction.
In this unit the gradient is a **measurement made while standing
still** at given weights; `C3-gradient-descent` will be the unit that
walks.

**The derivation (F4's chain rule, route by route).**
Fix the data and differentiate
$\mathrm{MSE}(w, b) = \frac1n \sum_i r_i^2$ with
$r_i = \sum_k X_{ik} w_k + b - y_i$, with respect to one chosen $w_j$:

1. *Outer stage, term $i$:* $\dfrac{\partial\, r_i^2}{\partial r_i} = 2 r_i$.
2. *Inner stage:* $\dfrac{\partial r_i}{\partial w_j} = X_{ij}$ — scan
   the sum $\sum_k X_{ik} w_k$; only the $k = j$ term contains $w_j$,
   and the targets $y_i$ are constants.
3. *Routes add:* every example $i$ is a separate route from $w_j$ to the
   loss.

$$\boxed{\;\frac{\partial\, \mathrm{MSE}}{\partial w_j}
   = \frac{2}{n} \sum_{i} r_i\, X_{ij}\;}
\qquad\qquad
\boxed{\;\frac{\partial\, \mathrm{MSE}}{\partial b}
   = \frac{2}{n} \sum_{i} r_i\;}$$

The bias formula is the same computation with inner derivative
$\partial r_i / \partial b = 1$ — the bias behaves exactly like a weight
on an invisible all-ones column.

**The sign-convention ledger (settle it once).**
F4 wrote the same score with residual $y_i - \hat y_i$ and got
$-\frac{2}{N}\sum_n r_n X_{nj}$.
Same mathematics: the minus sign lives either inside the residual (our
convention, $r_i = \hat y_i - y_i$) or in front of the sum (F4's).
Mixing the two conventions — flipping one but not the other — is the
classic way to compute a gradient that points exactly backwards
(Pitfall 2 below).

**Worked by hand** on Section 1's tiny data ($r = (1, -1, -1)$, $n = 3$):
$$\frac{\partial\,\mathrm{MSE}}{\partial w_0}
   = \tfrac{2}{3}\big(1\cdot 2 - 1\cdot 1 - 1\cdot 0\big) = \tfrac{2}{3},
\qquad
\frac{\partial\,\mathrm{MSE}}{\partial w_1}
   = \tfrac{2}{3}\big(1\cdot 1 - 1\cdot 3 - 1\cdot 2\big) = -\tfrac{8}{3},$$
$$\frac{\partial\,\mathrm{MSE}}{\partial b}
   = \tfrac{2}{3}\big(1 - 1 - 1\big) = -\tfrac{2}{3}.$$
Reading: at these knobs, increasing $w_1$ *decreases* the loss fastest
(most negative partial); the model would rather predict higher on
feature-1-heavy examples.

**In code** — broadcasting only, checked against central differences
(F4's checker; the checker MAY loop, graded functions may not):

In [ ]:
def mse_grad_w(X, y, w, b):
    """Shape-(d,) array of partials dMSE/dw_j. Broadcasting only."""
    r = predict(X, w, b) - y                      # (n,)
    return (2 / X.shape[0]) * (r[:, None] * X).sum(axis=0)


def mse_grad_b(X, y, w, b):
    """Scalar partial dMSE/db."""
    r = predict(X, w, b) - y
    return (2 / X.shape[0]) * r.sum()


# hand-checkable tiny case
print("tiny grad w:", mse_grad_w(X_tiny, y_tiny, w_tiny, b_tiny))  # (2/3, -8/3)
print("tiny grad b:", mse_grad_b(X_tiny, y_tiny, w_tiny, b_tiny))  # -2/3

# seeded larger case + central-difference check
rng = np.random.default_rng(SEED)
X5 = rng.normal(0, 1, (40, 3))
y5 = rng.normal(0, 1, 40)
w5 = np.array([1.0, -0.5, 2.0])
b5 = 0.5

g_w = mse_grad_w(X5, y5, w5, b5)
g_b = mse_grad_b(X5, y5, w5, b5)

h = 1e-6
check_gap = 0.0
for j in range(3):                                # checker MAY loop
    step = np.zeros(3)
    step[j] = h
    est = (mse(X5, y5, w5 + step, b5) - mse(X5, y5, w5 - step, b5)) / (2 * h)
    check_gap = max(check_gap, abs(est - g_w[j]))
est_b = (mse(X5, y5, w5, b5 + h) - mse(X5, y5, w5, b5 - h)) / (2 * h)
check_gap = max(check_gap, abs(est_b - g_b))

print("grad w:", g_w, "  grad b:", g_b)
print("check_gap:", check_gap)                    # ~1e-9

**Where the gradient is zero: residual geometry.**
At the *best* knob setting no nudge can improve the loss, so every
partial is zero:
$$\sum_i r_i X_{ij} = 0 \ \text{ for every } j,
\qquad \sum_i r_i = 0 .$$
In F2 language: **the residual vector is orthogonal to every feature
column and to the all-ones column** — orthogonal to every direction the
blend of Section 2's column picture can still move.
That is F2's projection picture at full scale: the best reachable
prediction vector is the *closest point* of the reachable set to $y$,
and the miss $r$ sticks out perpendicularly.

We can *watch* this without fitting anything: the weights below were
computed elsewhere and are **supplied** as givens (this unit's standing
arrangement — weight-finding machinery arrives in
`C3-gradient-descent`).

In [ ]:
# Supplied: best-fit weights for Section 4's 50-row dataset (computed
# elsewhere; here they are data, not something we derive).
w_best = np.array([1.945772, -0.746968])
b_best = 2.883274

r_best = predict(X4, w_best, b_best) - y4
print("r . column_0 / n:", (r_best * X4[:, 0]).mean())    # ~1e-7
print("r . column_1 / n:", (r_best * X4[:, 1]).mean())    # ~1e-7
print("mean residual   :", r_best.mean())                 # ~1e-7
print("gradient at best:", mse_grad_w(X4, y4, w_best, b_best))
print("MSE at best     :", mse(X4, y4, w_best, b_best))

Every dot product is numerically zero (up to the supplied weights'
rounding): standing at the best knobs, the residual has no component
along anything the model could still use.
Compare Section 4's scoreboard: candidate A's MSE was $\approx 0.40$ on
the train rows; the true optimum squeezes the *whole-dataset* MSE to
$\approx 0.31$ — and no linear knobs can do better, because the
remaining miss is perpendicular to every knob's direction.

### Checkpoint 5

1. By hand: $X = \begin{pmatrix}1 & 1\\ 2 & 0\end{pmatrix}$,
   $y = (1, 3)$, $w = (1, 1)$, $b = 0$.
   Compute $\hat y$, $r$, both weight partials, and the bias partial.
   (One of the three comes out $0$ — which, and what does that mean?)
2. A classmate defines $r_i = y_i - \hat y_i$ instead.
   Rewrite both boxed formulas in their convention.
3. At the best knob setting, what is $\sum_i r_i$, and which "invisible
   column" of the map picture is that orthogonality statement about?

## 6. Worked Exam-Style Example: Normal-Form Multiple Choice

Round 1 wraps numeric answers in a **normal form** so that exactly one
decoded option is right.
Here is one solved in full, in the real register.

---

**Problem (reasoning is not required; no code needed).**
For the dataset
$$X = \begin{pmatrix} 2 & 1 \\ 1 & 3 \end{pmatrix}, \qquad y = (1, 5),$$
the linear model $\hat y_i = \sum_k X_{ik} w_k + b$ is evaluated at
$w = (2, 1)$, $b = -1$, with
$\mathrm{MSE} = \frac{1}{n}\sum_i (\hat y_i - y_i)^2$.
Compute $\dfrac{\partial\,\mathrm{MSE}}{\partial w_0}$ at these weights.
Your answer is a nonzero integer that can be written as $s \cdot m$ with
$s \in \{+1, -1\}$ and $m$ a positive integer.
What is $s + m$?

A. 2  B. 4  C. 6  D. 9  E. 11

---

**Solution.**

*Step 1 — write the formula before touching numbers.*
$\dfrac{\partial\,\mathrm{MSE}}{\partial w_0}
 = \dfrac{2}{n} \sum_i r_i X_{i0}$ with $r_i = \hat y_i - y_i$, $n = 2$.

*Step 2 — predictions, bias included.*
$\hat y_0 = 2\cdot 2 + 1\cdot 1 - 1 = 4$;
$\hat y_1 = 1\cdot 2 + 3\cdot 1 - 1 = 4$.

*Step 3 — residuals.*
$r_0 = 4 - 1 = 3$; $r_1 = 4 - 5 = -1$.

*Step 4 — assemble.*
$\dfrac{\partial\,\mathrm{MSE}}{\partial w_0}
 = \tfrac{2}{2}\big(3 \cdot 2 + (-1) \cdot 1\big) = 5$.

*Step 5 — decode the normal form.*
$5 = s \cdot m$ with $s = +1$, $m = 5$, so $s + m = 6$: **answer C**.
The traps are engineered in: forget the bias and the residuals become
$(4, 0)$, giving $8 \to s + m = 9$ (option D); flip the residual
convention without flipping anything else and you get
$-5 \to s + m = 4$ (option B).
A wrong *process* lands on a *different* decoded option — that is what
makes the encoding gradable.

*Step 6 — the free cross-check* (whenever code is allowed): a central
difference must land on $5$.

In [ ]:
X6 = np.array([[2.0, 1.0], [1.0, 3.0]])
y6 = np.array([1.0, 5.0])
w6 = np.array([2.0, 1.0])
b6 = -1.0

hand = mse_grad_w(X6, y6, w6, b6)[0]
h = 1e-6
numeric = (mse(X6, y6, w6 + np.array([h, 0]), b6)
           - mse(X6, y6, w6 - np.array([h, 0]), b6)) / (2 * h)
print("hand:", hand, "  numeric:", numeric)
print("decode: s = +1, m = 5  ->  s + m =", 1 + 5)

### Checkpoint 6

1. Same data and weights, same normal form: compute
   $\partial\,\mathrm{MSE}/\partial b$ and decode $s + m$.
2. One sentence each: why does the $s \cdot m$ encoding force everyone
   who computed correctly onto the same option, and why must the
   statement guarantee the answer is nonzero?

## 7. Common Pitfalls I

**Pitfall 1 — forgetting the bias in the predictions.**
The shapes all work out, the code runs, and every prediction is wrong by
the same constant.
The tell: at supposedly-good weights the *mean residual* is far from
zero (Section 5 said it should vanish at the optimum — a good rule has
no systematic high/low bias).

In [ ]:
pred_broken = (X4 * w_best).sum(axis=1)            # BROKEN: dropped + b
pred_fixed = (X4 * w_best).sum(axis=1) + b_best

print("mean residual, broken:", (pred_broken - y4).mean())   # ~ -b_best
print("mean residual, fixed :", (pred_fixed - y4).mean())    # ~ 0
print("MSE broken:", np.mean((pred_broken - y4)**2))
print("MSE fixed :", np.mean((pred_fixed - y4)**2))

A constant miss of $b_{\text{best}} \approx 2.88$ costs
$\approx 2.88^2 \approx 8.3$ of MSE all by itself.
Habit: after computing residuals at supplied "good" weights, print
`r.mean()` — a value far from $0$ means a level problem, and the bias is
the level knob.

**Pitfall 2 — mixing the two residual conventions.**
Compute $r = y - \hat y$ (F4's convention) but keep this unit's
$+\frac{2}{n}$ formula, and the gradient comes out exactly negated —
pointing uphill while you believe it points downhill.
The central-difference checker exposes it instantly:

In [ ]:
r_conv_mixed = y5 - predict(X5, w5, b5)            # F4-convention residual...
grad_broken = (2 / 40) * (r_conv_mixed[:, None] * X5).sum(axis=0)  # BROKEN: + formula
grad_fixed = mse_grad_w(X5, y5, w5, b5)

num = np.zeros(3)
for j in range(3):
    step = np.zeros(3)
    step[j] = 1e-6
    num[j] = (mse(X5, y5, w5 + step, b5) - mse(X5, y5, w5 - step, b5)) / 2e-6

print("broken:", grad_broken, "  gap:", np.abs(grad_broken - num).max())
print("fixed :", grad_fixed, "  gap:", np.abs(grad_fixed - num).max())

Either convention is fine — *committed to consistently*.
This unit pins $r_i = \hat y_i - y_i$ with $+\frac{2}{n}$; F4 pinned the
mirror image.
Check the sign of one partial against a one-coordinate central
difference before trusting any gradient you wrote.

**Pitfall 3 — the wrong axis, and sum where mean belongs.**
Two proportional-or-shape bugs with one moral.
$\sum_i r_i X_{ij}$ sums over *examples* (axis 0), leaving one number
per *feature*; `axis=1` produces a shape-$(n,)$ array — the shape
contract catches it before any grading does.
And dropping the $1/n$ (using `.sum()` where `.mean()` belongs in the
loss) inflates everything by a factor of $n$ — "proportional to right"
is still graded wrong.

In [ ]:
r5 = predict(X5, w5, b5) - y5

wrong_axis = (2 / 40) * (r5[:, None] * X5).sum(axis=1)
print("wrong-axis shape:", wrong_axis.shape, " (contract says (3,))")
print("right-axis shape:", mse_grad_w(X5, y5, w5, b5).shape)

mse_sum_bug = np.sum(r5**2)                        # BROKEN: forgot the 1/n
print("sum-bug 'MSE':", mse_sum_bug, " = n x true MSE:", 40 * np.mean(r5**2))

Predict the shape before you run: $d$ knobs need $d$ partials.
And write the $\frac{2}{n}$ into the formula on paper before typing —
the factor you never wrote down is the factor you'll never miss.

### Checkpoint 7

1. A classmate's gradient passes the shape contract but every entry is
   exactly $-1$ times the checker's estimate.
   Which pitfall, and what is the one-line fix?
2. Another classmate's "MSE" is exactly $50$ times too big on a 50-row
   dataset. Which line of their code is wrong?
3. At supplied allegedly-optimal weights you find
   `r.mean() == 1.7`. Which knob is mis-set, and which pitfall's habit
   catches it?

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. $1 \cdot 3 + 4 \cdot (-1) + 2 = 1$.
2. `w`: `(6,)`; `b`: a scalar (shape `()`); $\hat y$: `(200,)`.
3. With $w_k = 0$ the model *ignores* feature $k$ entirely; Session 2 is
   about making the model set weights to exactly zero on purpose —
   automatic feature pruning.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. $\hat y = w_0 x + b$: $w_0$ is the slope, $b$ the intercept.
2. $\hat y = 2 c_0 + 3 c_1 + 1 \cdot \mathbf 1
   = (2, 0) + (0, 6) + (1, 1) = (3, 7)$.
3. $\hat y$ must be a blend of just $3$ vectors ($2$ feature columns
   plus the all-ones), while $y$ roams a $100$-dimensional space —
   almost every target is out of reach.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. $r = (2, -1)$; $\mathrm{MSE} = \frac{4 + 1}{2} = 2.5$.
2. Squaring stops opposite-signed misses from cancelling into a fake
   perfect score; and it punishes one large miss much harder than many
   small ones, so the loss cares about outliers.
3. $\mathrm{MSE} = \lVert \hat y - y \rVert^2 / n$.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. It predicts the same number, $\bar y_{\text{train}}$, for every
   example — the "no features" baseline.
   In code: `b_d = y_train.mean()`, then
   `mse(X_train, y_train, np.zeros(2), b_d)` ≈ 5.6 and
   `mse(X_test, y_test, np.zeros(2), b_d)` ≈ 3.9 — far above candidate
   A, as a rule that ignores every feature should be.
2. Because the train rows reward memorization as happily as
   understanding; only rows the candidate never touched measure how it
   generalizes.
3. False — a candidate can top the train column by fitting that sample's
   noise and lose the test column (overfitting; Session 2 §1 shows one).

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. $\hat y = (2, 2)$, $r = (1, -1)$;
   $\partial_{w_0} = \frac{2}{2}(1\cdot1 - 1\cdot2) = -1$;
   $\partial_{w_1} = \frac{2}{2}(1\cdot1 - 1\cdot0) = 1$;
   $\partial_b = \frac{2}{2}(1 - 1) = 0$.
   The bias partial vanishes: the residuals already average to zero, so
   shifting the level helps nothing — $b$ is (locally) perfectly set
   while the weights are not.
2. With $r_i = y_i - \hat y_i$:
   $\partial_{w_j} = -\frac{2}{n}\sum_i r_i X_{ij}$ and
   $\partial_b = -\frac{2}{n}\sum_i r_i$ — the minus migrates out front.
3. $\sum_i r_i = 0$; it is the orthogonality statement for the
   all-ones column that carries the bias.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. $\partial\,\mathrm{MSE}/\partial b = \frac{2}{2}(3 + (-1)) = 2$;
   decode $s = +1$, $m = 2$, so $s + m = 3$.
2. The decomposition of a nonzero integer into
   $(\text{sign}) \times (\text{positive size})$ is unique, so every
   correct computation decodes identically; $0$ has no sign, so a zero
   answer would make the decoding ambiguous — the statement must rule it
   out.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. Pitfall 2 (convention mixing): the residual and the formula disagree
   about where the minus lives.
   Fix one of them — e.g. compute `r = predict(...) - y` to match the
   $+\frac{2}{n}$ formula.
2. The loss line uses `np.sum(r**2)` where `np.mean(r**2)` belongs
   (Pitfall 3's second half).
3. The bias (level) knob — Pitfall 1's habit of printing `r.mean()` at
   supplied good weights flags it: near-optimal weights force the mean
   residual to $\approx 0$.

</details>